# How do you train with Keras?
**Topics:** model.compile() · model.fit() · model.evaluate() · model.predict() · Metrics · Loss Functions · Optimizers

---
## 1. model.compile()

### What it is
- Configures the model for training
- Must be called before `model.fit()`
- Sets optimizer, loss function, and evaluation metrics

### Key arguments
- `optimizer` — update rule: `'adam'`, `'sgd'`, or optimizer object
- `loss` — what to minimize: `'binary_crossentropy'`, `'sparse_categorical_crossentropy'`, etc.
- `metrics` — what to track: `['accuracy']`, `['mae']`, etc.

### Loss function cheatsheet

| Task | Loss | Label format |
|---|---|---|
| Binary classification | `binary_crossentropy` | 0 or 1 |
| Multi-class (int labels) | `sparse_categorical_crossentropy` | integers |
| Multi-class (one-hot) | `categorical_crossentropy` | one-hot vectors |
| Regression | `mse` or `mae` | float values |

### Gotchas
- Use `sparse_categorical_crossentropy` when labels are integers — avoids one-hot conversion
- `from_logits=True` when model output has no softmax/sigmoid — more numerically stable
- Metrics are reset at the start of each epoch automatically

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

# --- model.compile() examples ---
model = tf.keras.Sequential([
    tf.keras.layers.Dense(32, activation='relu', input_shape=(8,)),
    tf.keras.layers.Dense(3,  activation='softmax'),
])

# Standard compile
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

# With from_logits (no softmax in model)
model_logits = tf.keras.Sequential([
    tf.keras.layers.Dense(32, activation='relu', input_shape=(8,)),
    tf.keras.layers.Dense(3),   # no activation — raw logits
])
model_logits.compile(
    optimizer='adam',
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy'],
)

# Multi-metric compile
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy', tf.keras.metrics.TopKCategoricalAccuracy(k=2)],
)

# Visualize loss function behavior
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

p = np.linspace(0.01, 0.99, 200)
axes[0].plot(p, -np.log(p),     color='#2ECC71', linewidth=2.5, label='True label=1 (BCE)')
axes[0].plot(p, -np.log(1-p),   color='#E74C3C', linewidth=2.5, label='True label=0 (BCE)')
axes[0].set_title('Binary Cross-Entropy', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Predicted probability'); axes[0].set_ylabel('Loss')
axes[0].set_ylim(0, 5); axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)
axes[0].spines['top'].set_visible(False); axes[0].spines['right'].set_visible(False)

err = np.linspace(-3, 3, 300)
axes[1].plot(err, err**2,       color='#E74C3C', linewidth=2.5, label='MSE')
axes[1].plot(err, np.abs(err),  color='#3498DB', linewidth=2.5, label='MAE')
huber = np.where(np.abs(err)<=1, 0.5*err**2, np.abs(err)-0.5)
axes[1].plot(err, huber,        color='#2ECC71', linewidth=2.5, label='Huber')
axes[1].set_title('Regression Losses', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Prediction error'); axes[1].set_ylabel('Loss')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)
axes[1].spines['top'].set_visible(False); axes[1].spines['right'].set_visible(False)

plt.suptitle('Loss Functions in Keras', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('keras_losses.png', dpi=120, bbox_inches='tight')
plt.show()


---
## 2. model.fit()

### What it is
- Trains the model for a fixed number of epochs
- Returns a `History` object — contains loss and metric values per epoch
- Handles batching, shuffling, validation split automatically

### Key arguments
- `x, y` — training data
- `epochs` — number of full passes through data
- `batch_size` — samples per gradient update (default 32)
- `validation_split` — fraction of training data for validation
- `validation_data` — explicit `(X_val, y_val)` tuple
- `callbacks` — list of callback objects (EarlyStopping, etc.)
- `class_weight` — dict for imbalanced classes

### Gotchas
- `validation_split` takes from the END of the data — shuffle first
- `shuffle=True` by default — shuffles training data each epoch
- History object keys: `loss`, `val_loss`, `accuracy`, `val_accuracy`

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42); tf.random.set_seed(42)

X = np.random.randn(1000, 8).astype(np.float32)
y = (X[:, 0] + X[:, 1] > 0).astype(np.int32)

X_train, X_val = X[:800], X[800:]
y_train, y_val = y[:800], y[800:]

model = tf.keras.Sequential([
    tf.keras.layers.Dense(32, activation='relu', input_shape=(8,)),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(16, activation='relu'),
    tf.keras.layers.Dense(1,  activation='sigmoid'),
])
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

history = model.fit(
    X_train, y_train,
    epochs=40,
    batch_size=32,
    validation_data=(X_val, y_val),
    verbose=0,
)

print(f"Final train accuracy: {history.history['accuracy'][-1]:.3f}")
print(f"Final val accuracy  : {history.history['val_accuracy'][-1]:.3f}")
print(f"History keys        : {list(history.history.keys())}")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
epochs = range(1, len(history.history['loss'])+1)

axes[0].plot(epochs, history.history['loss'],     color='#3498DB', linewidth=2, label='Train loss')
axes[0].plot(epochs, history.history['val_loss'], color='#E74C3C', linewidth=2, label='Val loss', linestyle='--')
axes[0].set_title('Loss Curves', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].legend(fontsize=10); axes[0].grid(True, alpha=0.3)
axes[0].spines['top'].set_visible(False); axes[0].spines['right'].set_visible(False)

axes[1].plot(epochs, history.history['accuracy'],     color='#3498DB', linewidth=2, label='Train acc')
axes[1].plot(epochs, history.history['val_accuracy'], color='#E74C3C', linewidth=2, label='Val acc', linestyle='--')
axes[1].set_title('Accuracy Curves', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
axes[1].legend(fontsize=10); axes[1].grid(True, alpha=0.3)
axes[1].spines['top'].set_visible(False); axes[1].spines['right'].set_visible(False)

plt.suptitle('model.fit() Training History', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fit_history.png', dpi=120, bbox_inches='tight')
plt.show()


---
## 3. model.evaluate() & model.predict()

### What it is
- `model.evaluate()` — computes loss and metrics on a dataset, returns scalars
- `model.predict()` — runs forward pass, returns predictions as NumPy array

### Key differences

| Method | Returns | Use for |
|---|---|---|
| `model.evaluate()` | loss + metrics (scalars) | Final test set evaluation |
| `model.predict()` | predictions (array) | Getting outputs for downstream use |
| `model(x)` | tensor | Quick forward pass in code |

### Gotchas
- `model.predict()` always runs in inference mode — Dropout off, BN uses running stats
- For large datasets, `model.predict()` batches automatically — more memory efficient than `model(x)`
- `model.evaluate()` returns a list: `[loss, metric1, metric2, ...]`

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

np.random.seed(42); tf.random.set_seed(42)

X = np.random.randn(500, 8).astype(np.float32)
y = (X[:, 0] + X[:, 1] > 0).astype(np.int32)

model = tf.keras.Sequential([
    tf.keras.layers.Dense(32, activation='relu', input_shape=(8,)),
    tf.keras.layers.Dense(1, activation='sigmoid'),
])
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.fit(X[:400], y[:400], epochs=20, batch_size=32, verbose=0)

X_test, y_test = X[400:], y[400:]

# evaluate
results = model.evaluate(X_test, y_test, verbose=0)
print(f"evaluate() returns: {results}")
print(f"Test loss    : {results[0]:.4f}")
print(f"Test accuracy: {results[1]:.4f}")

# predict
probs = model.predict(X_test, verbose=0)
preds = (probs > 0.5).astype(int).ravel()
print(f"predict() shape: {probs.shape}")
print(f"Sample probs   : {probs[:5].ravel().round(3)}")

# Confusion matrix
cm = confusion_matrix(y_test, preds)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

im = axes[0].imshow(cm, cmap='Blues')
axes[0].set_xticks([0, 1]); axes[0].set_yticks([0, 1])
axes[0].set_xticklabels(['Pred 0', 'Pred 1'], fontsize=11)
axes[0].set_yticklabels(['True 0', 'True 1'], fontsize=11)
axes[0].set_title('Confusion Matrix', fontsize=12, fontweight='bold')
for i in range(2):
    for j in range(2):
        axes[0].text(j, i, str(cm[i,j]), ha='center', va='center',
                     fontsize=14, fontweight='bold', color='white' if cm[i,j]>cm.max()/2 else 'black')
plt.colorbar(im, ax=axes[0], shrink=0.8)

axes[1].hist(probs[y_test==0], bins=25, color='#E74C3C', alpha=0.7, label='True class 0')
axes[1].hist(probs[y_test==1], bins=25, color='#2ECC71', alpha=0.7, label='True class 1')
axes[1].axvline(0.5, color='black', linewidth=2, linestyle='--', label='Threshold 0.5')
axes[1].set_title('Predicted Probability Distribution', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Predicted probability'); axes[1].set_ylabel('Count')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)
axes[1].spines['top'].set_visible(False); axes[1].spines['right'].set_visible(False)

plt.suptitle('model.evaluate() and model.predict()', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('evaluate_predict.png', dpi=120, bbox_inches='tight')
plt.show()


---
## 4. Optimizers

### What it is
- Keras provides all standard optimizers in `tf.keras.optimizers`
- Can pass as string shortcut or as object with custom hyperparameters

### Common optimizers

| Optimizer | When to use |
|---|---|
| `Adam(lr=1e-3)` | Default for most tasks |
| `AdamW(lr=1e-3, weight_decay=1e-4)` | Transformers, better regularization |
| `SGD(lr=0.01, momentum=0.9)` | CV tasks, well-tuned pipelines |
| `RMSprop` | RNNs |

### Gotchas
- Pass `learning_rate` not `lr` to Keras optimizer objects
- Weight decay in Adam is coupled — use AdamW for proper L2 regularization
- `clipnorm` and `clipvalue` arguments add gradient clipping directly to optimizer

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42); tf.random.set_seed(42)

X = np.random.randn(500, 8).astype(np.float32)
y = (X[:, 0] + X[:, 1] > 0).astype(np.int32)

def train_with_optimizer(optimizer, epochs=40):
    model = tf.keras.Sequential([
        tf.keras.layers.Dense(32, activation='relu', input_shape=(8,)),
        tf.keras.layers.Dense(1, activation='sigmoid'),
    ])
    model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])
    history = model.fit(X[:400], y[:400], epochs=epochs,
                        validation_data=(X[400:], y[400:]), verbose=0)
    return history.history['val_accuracy']

configs = [
    (tf.keras.optimizers.SGD(learning_rate=0.01, momentum=0.9), '#E74C3C', 'SGD+Momentum'),
    (tf.keras.optimizers.Adam(learning_rate=1e-3),               '#3498DB', 'Adam'),
    (tf.keras.optimizers.RMSprop(learning_rate=1e-3),            '#2ECC71', 'RMSprop'),
]

fig, ax = plt.subplots(figsize=(10, 5))
for opt, color, label in configs:
    accs = train_with_optimizer(opt)
    ax.plot(accs, color=color, linewidth=2.5, label=label, alpha=0.9)

ax.set_xlabel('Epoch', fontsize=12); ax.set_ylabel('Validation Accuracy', fontsize=12)
ax.set_title('Optimizer Comparison', fontsize=13, fontweight='bold')
ax.legend(fontsize=11); ax.grid(True, alpha=0.3)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('keras_optimizers.png', dpi=120, bbox_inches='tight')
plt.show()

# Gradient clipping in optimizer
opt_clipped = tf.keras.optimizers.Adam(learning_rate=1e-3, clipnorm=1.0)
print(f"Optimizer with clipnorm: {opt_clipped.get_config()}")


### Interview Q&A

**What does `from_logits=True` mean?**
- Model output is raw logits (no sigmoid/softmax applied)
- Loss function applies the activation internally — numerically more stable
- Always prefer `from_logits=True` for classification losses

**How do you handle class imbalance in Keras?**
- `class_weight` parameter in `model.fit()`: `{0: 1.0, 1: 5.0}` — upweights minority class
- Or oversample minority class in your dataset pipeline

**What is the difference between `validation_split` and `validation_data`?**
- `validation_split=0.2` — Keras takes last 20% of training data as val (no shuffling before split)
- `validation_data=(X_val, y_val)` — explicit separate val set — always prefer this

### Gotchas
- `model.fit()` shuffles training data each epoch by default — good for SGD, disable for time series
- History loss is averaged over batches — val loss is computed on full val set at epoch end
- `verbose=0` silences output, `verbose=1` shows progress bar, `verbose=2` shows one line per epoch